# 06 — JAX Features: JIT, vmap, and Autodiff

This notebook explores the JAX-specific capabilities of `fewtrax` that distinguish it from a standard NumPy/SciPy EMRI implementation.

**Topics:**
- JIT compilation with `eqx.filter_jit` — first-call overhead vs. subsequent speed
- `jax.vmap` — batching waveform generation over parameter arrays
- `jax.grad` and `jax.jacobian` — differentiating through the trajectory ODE
- Fisher matrix computation for parameter estimation
- Brief outlook: HMC/NUTS sampling with `blackjax`

---

## Why JAX?

The EMRI waveform pipeline has three computationally intensive stages:

| Stage | fewtrax implementation | JAX benefit |
|---|---|---|
| Inspiral trajectory ODE | `diffrax` + `eqx.filter_jit` | JIT + GPU |
| Amplitude interpolation | `scipy.bisplev` (CPU loop) | *(currently CPU-only)* |
| Mode summation | `jax.numpy` matrix ops | JIT + GPU + vmap |

The trajectory and summation stages are fully JAX-native and benefit from all four JAX function transformations: `jit`, `vmap`, `grad`, and `pmap`.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import equinox as eqx
import time

jax.config.update("jax_enable_x64", True)

from dotenv import load_dotenv
load_dotenv()
DATA_DIR = os.getenv("FEW_DATA_DIR")
print(f"FEW_DATA_DIR = {DATA_DIR}")

from fewtrax.data import load_flux_data
from fewtrax.trajectory import EMRIInspiral

print(f"JAX version : {jax.__version__}")
print(f"Default backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

FEW_DATA_DIR = /Users/bertd/Documents/PhD/LISA/Codes/FastEMRIWaveforms/src/few/data
JAX version : 0.8.3
Default backend: cpu
Devices: [CpuDevice(id=0)]


## 6.1  JIT Compilation

JAX's JIT compiler traces Python functions into XLA computations on the first call ("tracing overhead"), then executes the compiled graph on all subsequent calls.

For `fewtrax`, the inspiral ODE is wrapped with `eqx.filter_jit` — an Equinox-aware JIT that handles `eqx.Module` pytrees (like `EMRIInspiral`) without re-tracing when only numeric values change.

The key distinction:
- **Static arguments** (array shapes, control flow structure) trigger recompilation if changed
- **Dynamic arguments** (numeric parameter values like `p0`, `e0`, `a`) never trigger recompilation

In [ ]:
flux_data = load_flux_data(DATA_DIR)

inspiral = EMRIInspiral(flux_data)

# --- First call: traces + compiles + executes ---
t0 = time.perf_counter()
t1, p1, e1, Pphi1, Pth1, Pr1 = inspiral(p0=12.0, e0=0.4, a=0.7, M=1e6, mu=10.0, T=0.5, dense_steps=150)
# Block until computation is complete
_ = p1.block_until_ready()
first_call = time.perf_counter() - t0

# --- Second call: runs compiled code, different parameters ---
t0 = time.perf_counter()
t2, p2, e2, Pphi2, Pth2, Pr2 = inspiral(p0=10.0, e0=0.3, a=0.7, M=1e6, mu=10.0, T=0.5, dense_steps=150)
_ = p2.block_until_ready()
second_call = time.perf_counter() - t0

# --- Third call ---
t0 = time.perf_counter()
t3, p3, e3, _, _, _ = inspiral(p0=14.0, e0=0.5, a=0.7, M=1e6, mu=10.0, T=0.5, dense_steps=150)
_ = p3.block_until_ready()
third_call = time.perf_counter() - t0

print(f"First  call (trace + compile): {first_call*1e3:.0f} ms")
print(f"Second call (compiled)       : {second_call*1e3:.1f} ms")
print(f"Third  call (compiled)       : {third_call*1e3:.1f} ms")
print(f"Speedup (1st → 2nd): {first_call/second_call:.0f}×")

In [ ]:
# Visualise the three trajectories
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for (t_arr, p_arr, e_arr, lab) in [
    (t1, p1, e1, "$p_0=12,\,e_0=0.4$"),
    (t2, p2, e2, "$p_0=10,\,e_0=0.3$"),
    (t3, p3, e3, "$p_0=14,\,e_0=0.5$"),
]:
    valid = ~jnp.isnan(p_arr)
    t_yr = jnp.asarray(t_arr)[valid] / (365.25 * 86400)
    axes[0].plot(t_yr, jnp.asarray(p_arr)[valid], label=lab)
    axes[1].plot(t_yr, jnp.asarray(e_arr)[valid], label=lab)

axes[0].set_xlabel("Time [yr]")
axes[0].set_ylabel("$p$ [$M$]")
axes[0].set_title("Semi-latus rectum")
axes[0].legend(fontsize=9)

axes[1].set_xlabel("Time [yr]")
axes[1].set_ylabel("$e$")
axes[1].set_title("Eccentricity")
axes[1].legend(fontsize=9)

plt.suptitle("Three trajectories — all run with same compiled code")
plt.tight_layout()
plt.show()

## 6.2  vmap: Batching Over Parameter Space

`jax.vmap` vectorises a function along a batch axis without writing explicit loops.  For EMRI parameter estimation, the key use cases are:

1. **Evaluating waveforms at a grid of parameter points** (for likelihood surfaces)
2. **Batching Fisher matrix columns** (computing all ∂h/∂θᵢ in one pass)
3. **Parallel MCMC chains** (each chain evaluates the waveform at a different point)

### How vmap works

```python
# Scalar function: scalar input → scalar output
f(x)  # x is a float

# Batched version: vector input → vector output
jax.vmap(f)(xs)  # xs has shape (N,)
```

Under the hood, vmap vectorises the XLA computation — it does NOT loop over the batch dimension.  On GPU this gives near-linear speedup with batch size.

In [ ]:
# Define a scalar function: (p0, e0) → final (p_end, e_end) after T=0.5 yr
def run_and_get_endpoint(p0: float, e0: float) -> jnp.ndarray:
    """Returns (p_end, e_end) at the end of the inspiral."""
    t_arr, p_arr, e_arr, _, _, _ = inspiral(p0=p0, e0=e0, a=0.7, M=1e6, mu=10.0, T=0.5)
    valid_mask = ~jnp.isnan(p_arr)
    n_valid = jnp.sum(valid_mask)
    p_end = jnp.where(valid_mask, p_arr, jnp.nan)
    e_end = jnp.where(valid_mask, e_arr, jnp.nan)
    return jnp.array([jnp.nanmin(p_end), e_end[n_valid - 1]])

# Batch over a grid of initial p0 values
p0_batch = jnp.linspace(10.0, 16.0, 8)
e0_fixed = jnp.full(8, 0.4)

batched_run = eqx.filter_jit(jax.vmap(run_and_get_endpoint))

t0 = time.perf_counter()
endpoints = batched_run(p0_batch, e0_fixed)
_ = endpoints.block_until_ready()
batch_first = time.perf_counter() - t0

t0 = time.perf_counter()
endpoints = batched_run(p0_batch, e0_fixed)
_ = endpoints.block_until_ready()
batch_second = time.perf_counter() - t0

print(f"Batch size: {len(p0_batch)}")
print(f"Batch first  call: {batch_first*1e3:.0f} ms")
print(f"Batch second call: {batch_second*1e3:.1f} ms")
print(f"Per-trajectory (compiled): {batch_second/len(p0_batch)*1e3:.2f} ms")
print(f"\nEndpoints (p_end, e_end):")
for i, p0 in enumerate(p0_batch):
    print(f"  p0={float(p0):.1f} → p_end={float(endpoints[i,0]):.3f}, e_end={float(endpoints[i,1]):.4f}")

In [ ]:
def get_inspiral_duration(p0: float, e0: float) -> float:
    """Returns inspiral time in years."""
    t_arr, p_arr, e_arr, _, _, _ = inspiral(p0=p0, e0=e0, a=0.7, M=1e6, mu=10.0, T=2.0)
    valid_mask = ~jnp.isnan(p_arr)
    n_valid = jnp.sum(valid_mask)
    t_end = t_arr[n_valid - 1]
    return t_end / (365.25 * 86400)

p0_grid = jnp.linspace(10.0, 18.0, 10)
e0_grid = jnp.linspace(0.05, 0.6, 10)

P0, E0 = jnp.meshgrid(p0_grid, e0_grid, indexing="ij")
p0_flat = P0.ravel()
e0_flat = E0.ravel()

batched_duration = eqx.filter_jit(jax.vmap(get_inspiral_duration))

print(f"Computing {len(p0_flat)} trajectories on a {len(p0_grid)}×{len(e0_grid)} grid…")
t0 = time.perf_counter()
durations_flat = batched_duration(p0_flat, e0_flat)
_ = durations_flat.block_until_ready()
elapsed = time.perf_counter() - t0
print(f"Wall time (incl. JIT): {elapsed:.2f} s ({elapsed/len(p0_flat)*1e3:.0f} ms per trajectory)")

t0 = time.perf_counter()
durations_flat = batched_duration(p0_flat, e0_flat)
_ = durations_flat.block_until_ready()
elapsed2 = time.perf_counter() - t0
print(f"Wall time (compiled) : {elapsed2:.2f} s ({elapsed2/len(p0_flat)*1e3:.1f} ms per trajectory)")

durations_2d = np.asarray(durations_flat).reshape(len(p0_grid), len(e0_grid))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(np.asarray(e0_grid), np.asarray(p0_grid), durations_2d, cmap="viridis")
plt.colorbar(im, ax=ax, label="Inspiral duration [yr]")
ax.set_xlabel("Initial eccentricity $e_0$")
ax.set_ylabel("Initial semi-latus rectum $p_0$ [$M$]")
ax.set_title("EMRI inspiral duration ($M=10^6 M_\\odot$, $\\mu=10 M_\\odot$, $a=0.7$)")
plt.tight_layout()
plt.show()

## 6.3  Automatic Differentiation Through the Trajectory

JAX's autodiff (`jax.grad`, `jax.jacfwd`, `jax.jacrev`) can differentiate through the entire `diffrax` ODE solver.  This is possible because:

1. `diffrax` implements adjoint-method backpropagation through ODE solutions
2. All geodesic functions (`kerr_geo_energy_equatorial`, `get_separatrix`, etc.) use `jax.lax.fori_loop` with static loop bounds — the only AD-compatible loop primitive in JAX
3. All JAX operations (`jnp.where`, `jnp.linalg.solve`, etc.) are differentiable

### Application: Sensitivity of orbital phase to initial conditions

The quantity $\partial \Phi_\phi / \partial \theta_i$ (gradient of accumulated orbital phase w.r.t. source parameters) is the central object in:
- **Fisher matrix** computations: $\Gamma_{ij} = (\partial_i h | \partial_j h) / S_n$
- **Matched filtering**: detecting parameter offsets that cause phase mismatch
- **Gradient-based samplers**: HMC/NUTS requires $\nabla \log p(\theta | d)$

In [ ]:
def total_phase_phi(p0: float, e0: float) -> float:
    """Total accumulated azimuthal phase Φφ at end of inspiral."""
    t_arr, p_arr, e_arr, Phi_phi, _, _ = inspiral(p0=p0, e0=e0, a=0.7, M=1e6, mu=10.0, T=0.5)
    valid_mask = ~jnp.isnan(p_arr)
    n_valid = jnp.sum(valid_mask)
    return Phi_phi[n_valid - 1]

grad_fn = eqx.filter_jit(jax.grad(total_phase_phi, argnums=(0, 1)))

p0_ref, e0_ref = 12.0, 0.4

print("Computing gradient of total phase Φφ w.r.t. initial conditions…")
t0 = time.perf_counter()
dPhi_dp0, dPhi_de0 = grad_fn(jnp.float64(p0_ref), jnp.float64(e0_ref))
_ = dPhi_dp0.block_until_ready()
elapsed = time.perf_counter() - t0

print(f"\n∂Φφ/∂p0 = {float(dPhi_dp0):.4e}  (rad / M)")
print(f"∂Φφ/∂e0 = {float(dPhi_de0):.4e}  (rad / unit-e)")
print(f"\nGradient computed in: {elapsed:.2f} s (first call includes JIT)")

dp = 0.01
de = 0.001

phi_ref = total_phase_phi(jnp.float64(p0_ref), jnp.float64(e0_ref))
phi_p   = total_phase_phi(jnp.float64(p0_ref + dp), jnp.float64(e0_ref))
phi_m   = total_phase_phi(jnp.float64(p0_ref - dp), jnp.float64(e0_ref))
phi_ep  = total_phase_phi(jnp.float64(p0_ref), jnp.float64(e0_ref + de))
phi_em  = total_phase_phi(jnp.float64(p0_ref), jnp.float64(e0_ref - de))

dPhi_dp0_fd = float((phi_p - phi_m) / (2 * dp))
dPhi_de0_fd = float((phi_ep - phi_em) / (2 * de))

print(f"\nFinite-difference cross-check:")
print(f"  ∂Φφ/∂p0: AD = {float(dPhi_dp0):.4e}, FD = {dPhi_dp0_fd:.4e}, "
      f"err = {abs(float(dPhi_dp0) - dPhi_dp0_fd)/abs(dPhi_dp0_fd):.2e}")
print(f"  ∂Φφ/∂e0: AD = {float(dPhi_de0):.4e}, FD = {dPhi_de0_fd:.4e}, "
      f"err = {abs(float(dPhi_de0) - dPhi_de0_fd)/abs(dPhi_de0_fd):.2e}")

## 6.4  Gradient as a Function of Parameters

The gradient $\partial \Phi_\phi / \partial p_0$ measures how sensitively the final phase depends on the initial separation.  In gravitational-wave data analysis this directly sets the parameter estimation uncertainty:

$$
\sigma_{p_0} \sim \left( \frac{1}{S_n} \left| \frac{\partial h}{\partial p_0} \right|^2 \right)^{-1/2}
\approx \left( \frac{(\mu/d_L)^2}{S_n} \left| \frac{\partial \Phi_\phi}{\partial p_0} \right|^2 \right)^{-1/2}
$$

Let's map out how this gradient varies across parameter space.

In [ ]:
grad_p0_fn = eqx.filter_jit(jax.grad(total_phase_phi, argnums=0))
grad_e0_fn = eqx.filter_jit(jax.grad(total_phase_phi, argnums=1))

p0_vals = jnp.linspace(10.0, 16.0, 12)
e0_vals = jnp.array([0.1, 0.3, 0.5])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for e0_v in e0_vals:
    dp_grads = []
    de_grads = []
    for p0_v in p0_vals:
        g_p = float(grad_p0_fn(jnp.float64(p0_v), jnp.float64(e0_v)))
        g_e = float(grad_e0_fn(jnp.float64(p0_v), jnp.float64(e0_v)))
        dp_grads.append(abs(g_p))
        de_grads.append(abs(g_e))

    axes[0].semilogy(np.asarray(p0_vals), dp_grads, "o-", label=f"$e_0={float(e0_v):.1f}$")
    axes[1].semilogy(np.asarray(p0_vals), de_grads, "o-", label=f"$e_0={float(e0_v):.1f}$")

axes[0].set_xlabel("$p_0$ [$M$]")
axes[0].set_ylabel(r"$|\partial \Phi_\phi / \partial p_0|$ [rad / $M$]")
axes[0].set_title("Phase sensitivity to initial separation")
axes[0].legend()

axes[1].set_xlabel("$p_0$ [$M$]")
axes[1].set_ylabel(r"$|\partial \Phi_\phi / \partial e_0|$ [rad]")
axes[1].set_title("Phase sensitivity to initial eccentricity")
axes[1].legend()

plt.suptitle("JAX autodiff through inspiral ODE ($M=10^6$, $\\mu=10$, $a=0.7$, $T=0.5$ yr)")
plt.tight_layout()
plt.show()

## 6.5  Fisher Matrix (Trajectory Phase)

The **Fisher information matrix** sets the Cramér–Rao lower bound on parameter estimation uncertainty.  In the large-SNR limit:

$$
\Gamma_{ij} = \left( \frac{\partial h}{\partial \theta_i} \,\middle|\, \frac{\partial h}{\partial \theta_j} \right)
= \int_0^\infty \frac{\tilde{h}_i^*(f)\, \tilde{h}_j(f)}{S_n(f)} \, df
$$

For a phase-dominated signal (the EMRI case), the dominant contribution comes from the phase gradient:

$$
\Gamma_{ij} \approx \text{SNR}^2 \times \left\langle \frac{\partial \Phi}{\partial \theta_i} \frac{\partial \Phi}{\partial \theta_j} \right\rangle_t
$$

Here we compute the Jacobian $\partial(\Phi_\phi, \Phi_r)_{\rm end} / \partial(p_0, e_0, a)$ using `jax.jacobian`.

In [ ]:
insp = EMRIInspiral(flux_data)

# Three-parameter function: (p0, e0, a) → (Φφ_end, Φr_end)
# With a as a call-time argument, we can now differentiate w.r.t. all three.

def phases_at_end(params):
    """Returns (Φφ_end, Φr_end) as a function of [p0, e0, a]."""
    p0, e0, a = params[0], params[1], params[2]
    _, p_arr, _, Phi_phi, _, Phi_r = insp(p0=p0, e0=e0, a=a, M=1e6, mu=10.0, T=0.5, dense_steps=150)
    valid_mask = ~jnp.isnan(p_arr)
    n_valid = jnp.sum(valid_mask)
    return jnp.array([Phi_phi[n_valid - 1], Phi_r[n_valid - 1]])

jac_fn = eqx.filter_jit(jax.jacobian(phases_at_end))

params_ref = jnp.array([12.0, 0.4, 0.7])

print("Computing Jacobian ∂(Φφ, Φr)/∂(p0, e0, a)…")
t0 = time.perf_counter()
J = jac_fn(params_ref)
_ = J.block_until_ready()
elapsed = time.perf_counter() - t0
print(f"Computed in {elapsed:.2f} s")

print(f"\nJacobian matrix:")
print(f"  ∂Φφ/∂p0 = {float(J[0,0]):.4e} rad/M")
print(f"  ∂Φφ/∂e0 = {float(J[0,1]):.4e} rad")
print(f"  ∂Φφ/∂a  = {float(J[0,2]):.4e} rad")
print(f"  ∂Φr/∂p0 = {float(J[1,0]):.4e} rad/M")
print(f"  ∂Φr/∂e0 = {float(J[1,1]):.4e} rad")
print(f"  ∂Φr/∂a  = {float(J[1,2]):.4e} rad")

# Fisher matrix (phase-only approximation, p0/e0/a subspace)
# Γ ≈ J^T J (for unit noise)
Fisher = J.T @ J
Cov = jnp.linalg.inv(Fisher)

print(f"\nFisher matrix (phase-only, unit noise):")
print(f"  Γ(p0,p0) = {float(Fisher[0,0]):.3e}")
print(f"  Γ(e0,e0) = {float(Fisher[1,1]):.3e}")
print(f"  Γ(a,a)   = {float(Fisher[2,2]):.3e}")
print(f"\nCramer-Rao lower bounds (1σ, phase-only):")
print(f"  σ(p0) ≥ {float(jnp.sqrt(Cov[0,0])):.3e} M")
print(f"  σ(e0) ≥ {float(jnp.sqrt(Cov[1,1])):.3e}")
print(f"  σ(a)  ≥ {float(jnp.sqrt(Cov[2,2])):.3e}")

## 6.6  Performance Summary

Let's benchmark the different execution modes to quantify the JAX advantage.

In [ ]:
import timeit

_ = insp(p0=12.0, e0=0.4, a=0.7, M=1e6, mu=10.0, T=0.5)[1].block_until_ready()

def run_single():
    return insp(p0=12.0, e0=0.4, a=0.7, M=1e6, mu=10.0, T=0.5)[1].block_until_ready()

n_reps = 20
t_single = timeit.timeit(run_single, number=n_reps) / n_reps * 1e3

p0_b = jnp.linspace(10.0, 16.0, 16)
e0_b = jnp.full(16, 0.4)
batch16 = eqx.filter_jit(jax.vmap(lambda p0, e0: insp(p0=p0, e0=e0, a=0.7, M=1e6, mu=10.0, T=0.5)[1]))
_ = batch16(p0_b, e0_b).block_until_ready()

def run_batch16():
    return batch16(p0_b, e0_b).block_until_ready()

t_batch16 = timeit.timeit(run_batch16, number=n_reps) / n_reps * 1e3

print(f"Single trajectory       : {t_single:.2f} ms")
print(f"Batch of 16 (vmapped)   : {t_batch16:.2f} ms")
print(f"Per-traj in batch       : {t_batch16/16:.2f} ms")
print(f"Batch speedup vs serial : {t_single * 16 / t_batch16:.1f}×")

labels = ["Single (JIT)", "Batch/16 (vmap)"]
times  = [t_single, t_batch16 / 16]

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(labels, times, color=["C0", "C2"])
ax.bar_label(bars, labels=[f"{t:.2f} ms" for t in times])
ax.set_ylabel("Time per trajectory [ms]")
ax.set_title("Trajectory timing: JIT vs vmap")
plt.tight_layout()
plt.show()

## 6.7  Outlook: HMC/NUTS Sampling

The combination of `jax.grad` through the trajectory and `jax.vmap` for parallel chains makes `fewtrax` directly compatible with gradient-based MCMC samplers.

### With `blackjax`

```python
import blackjax

# Log-posterior: log p(θ | d) ∝ -½ (h(θ) - d | h(θ) - d) / Sn
def log_posterior(params):
    p0, e0 = params["p0"], params["e0"]
    hp, hx = wf(M=1e6, mu=10.0, a=0.7, p0=p0, e0=e0, ...)
    residual = hp - data_hp
    return -0.5 * jnp.sum(residual**2 / noise_psd)

# NUTS sampler — requires ∇ log p(θ | d) automatically via JAX
nuts = blackjax.nuts(log_posterior, step_size=0.01, inverse_mass_matrix=...)

# Run N chains in parallel with vmap
vmap_nuts_step = jax.vmap(nuts.step)
```

### Amplitude backends

`fewtrax` provides two amplitude interpolation backends:

| Backend | JAX support | Notes |
|---|---|---|
| `AmplitudeInterpolator` | ✗ (scipy CPU loop) | Legacy; used for validation |
| `JAXAmplitudeInterpolator` | ✓ (jit/vmap/grad/GPU) | Default in `KerrEccentricEquatorialWaveform` |

`JAXAmplitudeInterpolator` evaluates all modes in a single fused `einsum` via JAX-native tricubic B-spline evaluation, replacing the per-mode `scipy.bisplev` loop.  Construct it with `load_amplitude_data_jax` and `JAXAmplitudeInterpolator` from `fewtrax.amplitude`.

In [ ]:
# Summary of JAX transformation support in fewtrax
print("JAX transformation support in fewtrax:")
print()
print(f"{'Component':<40} {'jit':>6} {'vmap':>6} {'grad':>6} {'GPU':>6}")
print("-" * 65)
rows = [
    ("Geodesic functions",              "✓", "✓", "✓", "✓"),
    ("Separatrix (bisection)",           "✓", "✓", "✓", "✓"),
    ("Fundamental frequencies",          "✓", "✓", "✓", "✓"),
    ("EMRIInspiral (diffrax ODE)",       "✓", "✓", "✓", "✓"),
    ("AmplitudeInterpolator.evaluate",   "✗", "✗", "✗", "✗"),
    ("JAXAmplitudeInterpolator",         "✓", "✓", "✓", "✓"),
    ("get_ylms_for_modes",               "✓", "✓", "✓", "✓"),
    ("direct_mode_sum",                  "✓", "✓", "✓", "✓"),
    ("interpolated_mode_sum",            "✓", "✓", "~", "✓"),
]
for name, jit, vmap, grad, gpu in rows:
    print(f"{name:<40} {jit:>6} {vmap:>6} {grad:>6} {gpu:>6}")
print()
print("~ = partial support (gradient through spline interpolation, not amplitude)")
print("✗ = not supported (scipy bisplev CPU call)")
print("  → JAXAmplitudeInterpolator is the default backend in KerrEccentricEquatorialWaveform")

## Summary

| JAX feature | fewtrax use case | Status |
|---|---|---|
| `eqx.filter_jit` | Inspiral ODE, mode summation | ✓ Fully supported |
| `jax.vmap` | Batch trajectory + phase evaluation | ✓ Trajectory fully vmapped |
| `jax.grad` | Phase gradients for Fisher matrix | ✓ Through ODE |
| `jax.jacobian` | Multi-output parameter sensitivity | ✓ Through ODE |
| End-to-end `grad` | Full waveform likelihood gradient | ✗ Blocked by `bisplev` |
| GPU execution | Full pipeline | ✗ Blocked by `bisplev` |

**The key next step** for full JAX-native operation is replacing `scipy.bisplev` in `AmplitudeInterpolator` with a JAX implementation of bicubic B-spline evaluation.  Once complete, the entire waveform — from physical parameters to $h_+, h_\times$ — will be JIT-compiled, vmappable, and differentiable end-to-end.

---

**End of tutorial series.**

| Notebook | Topic |
|---|---|
| [01](01_kerr_geodesics.ipynb) | Kerr geodesics: $E$, $L_z$, separatrix, frequencies |
| [02](02_emri_trajectory.ipynb) | EMRI inspiral ODE with diffrax |
| [03](03_amplitude_interpolation.ipynb) | Teukolsky amplitude interpolation |
| [04](04_mode_summation.ipynb) | Mode summation and spin-weighted harmonics |
| [05](05_full_waveform.ipynb) | Full `KerrEccentricEquatorialWaveform` API |
| **[06](06_jax_features.ipynb)** | **JIT, vmap, autodiff, Fisher matrix** |